In [51]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import ParameterGrid
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.base import BaseEstimator, TransformerMixin
from tqdm import tqdm
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor
from scipy.stats import t
import matplotlib.pyplot as plt
import matplotlib
from sklearn.pipeline import Pipeline

from statsmodels.stats.sandwich_covariance import cov_hac #heteroscedasticity and autocorrelation robust covariance matrix (Newey-West)
from statsmodels.regression.linear_model import OLS
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.stattools import durbin_watson
import statsmodels.api as sm

In [52]:
df_final_uk = pd.read_excel("data/processed/df_final_uk.xlsx")

In [53]:
# Convertir la colonne Date en Period mensuel
df_final_uk["Date"] = pd.to_datetime(df_final_uk["Date"], dayfirst=True, errors="coerce").dt.to_period("M")

# Ensuite, filtrer
df_final_uk = df_final_uk[df_final_uk["Date"] >= pd.Period("1995-07", freq="M")].reset_index(drop=True)
df_final_uk = df_final_uk.sort_values("Date").reset_index(drop=True) 

In [54]:
nan_percent = df_final_uk.isna().mean().sort_values(ascending=False) * 100
print(nan_percent)

rd_mve           19.078709
cashpr            2.593399
dy                2.330432
sp                2.221618
ep                2.221618
invest            2.121872
chsh              1.840769
agr               1.686616
chinv             1.686616
depr              1.251360
stdturn           0.870511
mvel1             0.852376
turn              0.852376
beta              0.725426
beta_squared      0.725426
mom36m            0.344577
idiovol           0.054407
mom12m            0.045339
chmom             0.045339
baspread          0.018136
retvol            0.018136
illiq             0.018136
indmom            0.000000
mom1m             0.000000
mom6m             0.000000
dolvol            0.000000
Date              0.000000
Ticker            0.000000
maxret            0.000000
excess_return     0.000000
dtype: float64


In [55]:
df_final_uk = df_final_uk.drop(columns=["rd_mve"])

In [56]:
df_final_uk = df_final_uk[~df_final_uk["Ticker"].isin(["SPX", "WEIR"])] #après normalisation : était encore vide donc trop de NaN on préfère les enlever
df_final_uk.reset_index(drop=True, inplace=True)


In [57]:
n_unique_tickers = df_final_uk["Ticker"].nunique()
print(f"Nombre de tickers uniques : {n_unique_tickers}")

Nombre de tickers uniques : 34


In [58]:
covariates = ["dolvol", "maxret", "retvol", "mom36m", "mom12m", "mom6m", "mom1m", "chmom", "turn", "indmom", "baspread", "illiq", "stdturn", "beta", "beta_squared",
"idiovol", "mvel1", "agr", "cashpr", "chinv", "chsh", "depr", "dy", "ep", "invest", "sp"]

In [59]:
print(len(covariates))

26


In [60]:
def generate_time_splits(df, date_col='Date',
                                   val_months=12,
                                   test_months=12,
                                   step_months=12,
                                   min_train_months=192):
    """
    Génère des splits temporels:
    - Train cumulatif (augmente d'un an à chaque refit)
    - Train 306 mois (= 85% de la data)
    - Validation = fenêtre fixe glissante de 1 an
    - Test = 1 an 
    - Avance de step_months à chaque itération : 1 an

    Paramètres :
    - df : DataFrame trié par date
    - date_col : nom de la colonne des dates
    - val_months : taille de la validation
    - test_months : taille du test 
    - step_months : pas de glissement
    - min_train_months : nombre minimum de mois de train initial

    Retour :
    - splits : liste de tuples (train_idx, val_idx, test_idx)
    """

    #On coupe chronologiquement donc on trie par date 
    df = df.sort_values(date_col).reset_index(drop=True)
    dates = sorted(df[date_col].unique())
    total_months = len(dates)

    splits = []

    #On démarre après avoir au moins min_train_months pour le train
    start = min_train_months
    while True:
        train_end = start  # train va de 0 jusqu'à train_end
        val_start = train_end
        val_end = val_start + val_months
        test_start = val_end
        test_end = test_start + test_months

        #Stop quand on a plus assez pour test
        if test_end > total_months:
            break

        train_dates = dates[:train_end]  
        val_dates = dates[val_start:val_end]
        test_dates = dates[test_start:test_end]

        train_idx = df[df[date_col].isin(train_dates)].index.tolist()
        val_idx = df[df[date_col].isin(val_dates)].index.tolist()
        test_idx = df[df[date_col].isin(test_dates)].index.tolist()

        splits.append((train_idx, val_idx, test_idx))

        #Décale fenêtre de un → on réactualise tous les 1 ans
        start += step_months

    return splits

In [61]:
def preprocess_split(X_train, X_val, X_test, covariates):
    """
    Impute les NaN par moyenne par Ticker (fit sur train),
    puis normalise chaque covariable entre -1 et 1 par date (rang cross-sectionnel).

    Paramètres
    ----------
    X_train, X_val, X_test : DataFrames bruts (avec 'Ticker' et 'Date')
    covariates : liste des colonnes numériques à traiter

    Retour
    ------
    X_train_scaled, X_val_scaled, X_test_scaled : DataFrames transformés (covariates seulement)
    """
    #Gestion des NaN : moyenne par Ticker calculée sur le train
    means_by_ticker = X_train.groupby("Ticker")[covariates].mean(numeric_only=True)

    def fill_na_with_means(df):
        df = df.copy()
        for col in covariates:
            #On remplace NaN par la moyenne du ticker
            df[col] = df.apply(
                lambda row: means_by_ticker[col][row["Ticker"]] 
                            if pd.isna(row[col]) and row["Ticker"] in means_by_ticker.index 
                            else row[col],
                axis=1
            )
        return df

    x_train_imp = fill_na_with_means(x_train)
    x_val_imp   = fill_na_with_means(x_val)
    x_test_imp  = fill_na_with_means(x_test)

    #Normalisation cross-sectionnelle : par date
    def normalize_by_date(df):
        df = df.copy()
        out = pd.DataFrame(index=df.index, columns=covariates)
        for date_key, group_idx in df.groupby("Date").groups.items():
            sub = df.loc[group_idx, covariates]
            for cov in covariates:
                temp = sub[cov].dropna().sort_values()
                n = len(temp)
                if n == 1:
                    scores = pd.Series([0.0], index=temp.index)
                else:
                    scores = pd.Series(
                        2 * np.arange(n) / (n - 1) - 1, index=temp.index
                    )
                out.loc[temp.index, cov] = scores
        return out.astype(float)

    x_train_scaled = normalize_by_date(x_train_imp)
    x_val_scaled   = normalize_by_date(x_val_imp)
    x_test_scaled  = normalize_by_date(x_test_imp)

    #garde l'information de la date et du ticker
    x_train_scaled = pd.concat([x_train_imp[['Ticker','Date']].reset_index(drop=True), x_train_scaled.reset_index(drop=True)], axis=1)
    x_val_scaled   = pd.concat([x_val_imp[['Ticker','Date']].reset_index(drop=True), x_val_scaled.reset_index(drop=True)], axis=1)
    x_test_scaled  = pd.concat([x_test_imp[['Ticker','Date']].reset_index(drop=True), x_test_scaled.reset_index(drop=True)], axis=1)

    return x_train_scaled, x_val_scaled, x_test_scaled

In [62]:
#Permet de récupérer x et y > on exclut notre variable cible y et on garder les covariates
def get_x_y(df, idx, target="excess_return"):
    subset = df.loc[idx].copy()
    x = subset.drop(columns=[target]) 
    y = subset[target]
    return x, y

In [63]:
def get_y_ha(df, idx, col="ha_global"):
    subset = df.loc[idx].copy()
    y = subset[col]
    return y

In [64]:
#On découpe les splits puis on applique la gestion des NaN et la normalisation définie plus haut
splits = generate_time_splits(df_final_uk)

preprocessed_splits = []

for train_idx, val_idx, test_idx in tqdm(splits):
    x_train, y_train = get_x_y(df_final_uk, train_idx)
    x_val, y_val = get_x_y(df_final_uk, val_idx)
    x_test, y_test = get_x_y(df_final_uk, test_idx)

    #Imputation + Normalisation
    x_train, x_val, x_test = preprocess_split(x_train, x_val, x_test, covariates) #on enlève ticker et date

    preprocessed_splits.append((x_train, y_train, x_val, y_val, x_test, y_test))

  0%|          | 0/8 [00:00<?, ?it/s]

100%|██████████| 8/8 [01:14<00:00,  9.27s/it]


In [65]:
# 1. Calcul du ha_global
df_sorted = df_final_uk.sort_values(["Ticker", "Date"]).copy()
df_sorted["excess_return_shifted"] = df_sorted.groupby("Ticker")["excess_return"].shift(1)
cumulative_sum = df_sorted["excess_return_shifted"].cumsum()
count = (~df_sorted["excess_return_shifted"].isna()).cumsum()
df_sorted["ha_global"] = cumulative_sum / count

# 2. Mapping dans df_final
ha_by_date = df_sorted.dropna(subset=["ha_global"]).drop_duplicates("Date", keep="last").set_index("Date")["ha_global"]
df_final_uk["ha_global"] = df_final_uk["Date"].map(ha_by_date)

# 3. Table finale
df_ha = df_final_uk[["Date", "Ticker", "ha_global"]].rename(columns={"ha_global": "y_pred_ha"})
df_ha = df_ha.set_index(["Date", "Ticker"])


In [66]:
#preprocess mais pour ha 

y_trainval_ha_all = []
y_true_ha_all = []

for split_idx, (train_idx, val_idx, test_idx) in enumerate(splits, start=1):
    y_train = get_y_ha(df_final_uk, train_idx)
    y_val = get_y_ha(df_final_uk, val_idx)
    y_test = get_y_ha(df_final_uk, test_idx)

    # Concat trainval
    y_trainval = pd.concat([y_train, y_val])
    y_trainval_ha_all.append(y_trainval)
    y_true_ha_all.append(y_test)


In [67]:
"""Fonctions pour nos métriques : 
def r2: mesure le r2 selon la définition de Gu et al 
% ratio : success ratio, semblable à ce qui est fait dans le papier de Xiu et Liu
R2 benchmark : on compare le R2 de nos modèles à l'historical average 
"""

#r2
def r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum(y_true**2) 
    return 1 - ss_res/ss_tot if ss_tot != 0 else np.nan

#% ratio:
def success_ratio(y_true, y_pred, ignore_zero=True):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()

    sign_true = np.sign(y_true)
    sign_pred = np.sign(y_pred)

    if ignore_zero:
        mask = sign_true != 0
        sign_true = sign_true[mask]
        sign_pred = sign_pred[mask]

    if len(sign_true) == 0:
        return np.nan  
    return (sign_true == sign_pred).mean()

#R2 benchmark 
def r2_vs_benchmark(y_true, y_pred_model, y_pred_bench):
    T = len(y_true)
    mspe_model = (1/T) * np.sum((y_true - y_pred_model)**2)
    mspe_bench = (1/T) * np.sum((y_true - y_pred_bench)**2)
    return 1 - (mspe_model / mspe_bench)

In [68]:
# 1. Calcul du ha_global
df_sorted = df_final_uk.sort_values(["Ticker", "Date"]).copy()
df_sorted["excess_return_shifted"] = df_sorted.groupby("Ticker")["excess_return"].shift(1)
cumulative_sum = df_sorted["excess_return_shifted"].cumsum()
count = (~df_sorted["excess_return_shifted"].isna()).cumsum()
df_sorted["ha_global"] = cumulative_sum / count

# 2. Mapping final dans df_final
ha_by_date = df_sorted.dropna(subset=["ha_global"]).drop_duplicates("Date", keep="last").set_index("Date")["ha_global"]
df_final_uk["ha_global"] = df_final_uk["Date"].map(ha_by_date)

# 3. Création des tableaux prédictions
df_in_ha, df_oos_ha = [], []

for split_idx, (train_idx, val_idx, test_idx) in enumerate(splits, start=1):
    # Train + Val
    y_trainval = df_final_uk.loc[list(set(train_idx) | set(val_idx))]  # union
    df_in_split = y_trainval[["Date", "Ticker", "ha_global"]].copy()
    df_in_split["Split"] = split_idx
    df_in_ha.append(df_in_split)

    # Test
    y_test = df_final_uk.loc[test_idx]
    df_oos_split = y_test[["Date", "Ticker", "ha_global"]].copy()
    df_oos_split["Split"] = split_idx
    df_oos_ha.append(df_oos_split)

# 4. Concaténation + suppression des doublons
df_in_ha = pd.concat(df_in_ha, ignore_index=True)
df_in_ha.drop_duplicates(subset=["Date", "Ticker"], inplace=True)

df_oos_ha = pd.concat(df_oos_ha, ignore_index=True)

# 5. Renommage pour cohérence
df_in_ha = df_in_ha.rename(columns={"ha_global": "y_pred_ha"})
df_oos_ha = df_oos_ha.rename(columns={"ha_global": "y_pred_ha"})

# 6. Extraction des arrays finaux
y_trainval_ha = df_in_ha["y_pred_ha"].values
y_true_ha = df_oos_ha["y_pred_ha"].values

print(f"✅ Shapes HA — IN: {y_trainval_ha.shape} | OOS: {y_true_ha.shape}")


✅ Shapes HA — IN: (9792,) | OOS: (3264,)


In [69]:
#OLS

r2_in_ols, r2_oos_ols = [], []
success_ratio_in_ols, success_ratio_oos_ols = [], []
df_in_ols, df_oos_ols = [], []
feature_importance_ols = []

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):

    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])
    tickers_trainval = pd.concat([x_train["Ticker"], x_val["Ticker"]], ignore_index=True)
    dates_trainval = pd.concat([x_train["Date"], x_val["Date"]], ignore_index=True)

    assert len(x_trainval) == len(y_trainval) == len(tickers_trainval) == len(dates_trainval), f" Split {split_idx} — taille mismatch dans trainval"

    ols = LinearRegression()
    ols.fit(x_trainval, y_trainval)

    y_trainval_pred = ols.predict(x_trainval)
    y_test_pred = ols.predict(x_test[covariates])

    assert len(y_trainval_pred) == len(y_trainval), f" Split {split_idx} — pred trainval mismatch"
    assert len(y_test_pred) == len(y_test), f" Split {split_idx} — pred test mismatch"

    r2_in = r2(y_trainval, y_trainval_pred)
    r2_out = r2(y_test, y_test_pred)
    sr_in = success_ratio(y_trainval, y_trainval_pred)
    sr_out = success_ratio(y_test, y_test_pred)

    r2_in_ols.append(r2_in)
    r2_oos_ols.append(r2_out)
    success_ratio_in_ols.append(sr_in)
    success_ratio_oos_ols.append(sr_out)

    feature_importance_ols.append(np.abs(ols.coef_))

    df_in_split = pd.DataFrame({
        "Date": dates_trainval,
        "Ticker": tickers_trainval,
        "y_true": y_trainval.values,
        "y_pred_ols": y_trainval_pred,
        "Split": split_idx
    })
    df_in_ols.append(df_in_split)

    df_test_split = pd.DataFrame({
        "Date": x_test["Date"].reset_index(drop=True),
        "Ticker": x_test["Ticker"].reset_index(drop=True),
        "y_true": y_test.reset_index(drop=True),
        "y_pred_ols": y_test_pred
    })
    df_oos_ols.append(df_test_split)

    print(f"Split {split_idx}  R² IN: {r2_in:.6f} | OOS: {r2_out:.6f} | SR IN: {sr_in:.3f} | SR OOS: {sr_out:.3f}")

# Concat final
df_in_ols = pd.concat(df_in_ols, ignore_index=True)
df_in_ols.drop_duplicates(subset=["Date", "Ticker"], inplace=True)
df_oos_ols = pd.concat(df_oos_ols, ignore_index=True)

# Conversion
y_trainval_ols = df_in_ols["y_pred_ols"].values
y_true_ols = df_in_ols["y_true"].values
y_pred_oos_ols = df_oos_ols["y_pred_ols"]
y_true_oos_ols = df_oos_ols["y_true"].values

assert len(y_trainval_ols) == len(y_true_ols), "Taille mismatch entre y_trainval_ols et y_true_ols"
assert len(y_pred_oos_ols) == len(y_true_oos_ols), " Taille mismatch entre y_pred_oos_ols et y_true_oos_ols"


print(f"OLS R² IN global : {r2(y_true_ols, y_trainval_ols):.5f}")
print(f"OLS R² OOS global : {r2(y_true_oos_ols, y_pred_oos_ols):.5f}")



100%|██████████| 8/8 [00:00<00:00, 59.99it/s]

Split 1  R² IN: 0.017687 | OOS: 0.147330 | SR IN: 0.539 | SR OOS: 0.669
Split 2  R² IN: 0.021485 | OOS: 0.017651 | SR IN: 0.549 | SR OOS: 0.529
Split 3  R² IN: 0.021340 | OOS: 0.040772 | SR IN: 0.549 | SR OOS: 0.564
Split 4  R² IN: 0.021930 | OOS: -0.009295 | SR IN: 0.552 | SR OOS: 0.528
Split 5  R² IN: 0.021041 | OOS: 0.017663 | SR IN: 0.550 | SR OOS: 0.522
Split 6  R² IN: 0.021199 | OOS: 0.020417 | SR IN: 0.550 | SR OOS: 0.539
Split 7  R² IN: 0.021100 | OOS: -0.013567 | SR IN: 0.551 | SR OOS: 0.529
Split 8  R² IN: 0.020207 | OOS: -0.019138 | SR IN: 0.548 | SR OOS: 0.542


OLS R² IN global : 0.02093
OLS R² OOS global : 0.01808


In [70]:
#PLS 
r2_in_pls, r2_oos_pls = [], []
success_ratio_in_pls, success_ratio_oos_pls = [], []
df_in_pls, df_oos_pls = [], []
best_components_pls, mse_val_grids = [], []

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):

    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])
    tickers_trainval = pd.concat([x_train["Ticker"], x_val["Ticker"]], ignore_index=True)
    dates_trainval = pd.concat([x_train["Date"], x_val["Date"]], ignore_index=True)

    assert len(x_trainval) == len(y_trainval) == len(tickers_trainval) == len(dates_trainval), f" Split {split_idx} — taille mismatch dans trainval"

    # Sélection du meilleur k
    candidate_ks = np.arange(1, 26)
    mse_val_grid = []
    best_mse = float('inf')
    best_k = None

    for k in candidate_ks:
        pls = PLSRegression(n_components=k, scale=False)
        pls.fit(x_train[covariates], y_train)
        y_val_pred = pls.predict(x_val[covariates]).ravel()
        mse_val = mean_squared_error(y_val, y_val_pred)
        mse_val_grid.append(mse_val)
        if mse_val < best_mse:
            best_mse = mse_val
            best_k = k

    best_components_pls.append(best_k)
    mse_val_grids.append(mse_val_grid)

    # Réentraînement final
    pls_final = PLSRegression(n_components=best_k, scale=False)
    pls_final.fit(x_trainval, y_trainval)

    y_trainval_pred = pls_final.predict(x_trainval).ravel()
    y_test_pred = pls_final.predict(x_test[covariates]).ravel()

    assert len(y_trainval_pred) == len(y_trainval), f"Split {split_idx} — pred trainval mismatch"
    assert len(y_test_pred) == len(y_test), f"Split {split_idx} — pred test mismatch"

    r2_in = r2(y_trainval, y_trainval_pred)
    r2_out = r2(y_test, y_test_pred)
    sr_in = success_ratio(y_trainval, y_trainval_pred)
    sr_out = success_ratio(y_test, y_test_pred)

    r2_in_pls.append(r2_in)
    r2_oos_pls.append(r2_out)
    success_ratio_in_pls.append(sr_in)
    success_ratio_oos_pls.append(sr_out)

    df_in_split = pd.DataFrame({
        "Date": dates_trainval,
        "Ticker": tickers_trainval,
        "y_true": y_trainval.values,
        "y_pred_pls": y_trainval_pred,
        "Split": split_idx
    })
    df_in_pls.append(df_in_split)

    df_test_split = pd.DataFrame({
        "Date": x_test["Date"].reset_index(drop=True),
        "Ticker": x_test["Ticker"].reset_index(drop=True),
        "y_true": y_test.reset_index(drop=True),
        "y_pred_pls": y_test_pred
    })
    df_oos_pls.append(df_test_split)

    print(f"Split {split_idx}  R² IN: {r2_in:.6f} | OOS: {r2_out:.6f} | SR IN: {sr_in:.3f} | SR OOS: {sr_out:.3f}")

# Concat final
df_in_pls = pd.concat(df_in_pls, ignore_index=True)
df_in_pls.drop_duplicates(subset=["Date", "Ticker"], inplace=True)
df_oos_pls = pd.concat(df_oos_pls, ignore_index=True)

# Conversion
y_trainval_pls = df_in_pls["y_pred_pls"].values
y_true_pls = df_in_pls["y_true"].values
y_pred_oos_pls = df_oos_pls["y_pred_pls"].values
y_true_oos_pls = df_oos_pls["y_true"].values

assert len(y_trainval_pls) == len(y_true_pls), "Taille mismatch entre y_trainval_pls et y_true_pls"
assert len(y_pred_oos_pls) == len(y_true_oos_pls), "Taille mismatch entre y_pred_oos_pls et y_true_oos_pls"

print(f"PLS R² IN global : {r2(y_true_pls, y_trainval_pls):.5f}")
print(f"PLS R² OOS global : {r2(y_true_oos_pls, y_pred_oos_pls):.5f}")


 12%|█▎        | 1/8 [00:00<00:04,  1.40it/s]

Split 1  R² IN: 0.013738 | OOS: 0.153359 | SR IN: 0.540 | SR OOS: 0.691


 25%|██▌       | 2/8 [00:01<00:04,  1.31it/s]

Split 2  R² IN: 0.020509 | OOS: 0.015766 | SR IN: 0.549 | SR OOS: 0.532


 38%|███▊      | 3/8 [00:02<00:04,  1.21it/s]

...

 88%|████████▊ | 7/8 [00:06<00:01,  1.10s/it]

Split 7  R² IN: 0.020217 | OOS: -0.009401 | SR IN: 0.551 | SR OOS: 0.520


100%|██████████| 8/8 [00:08<00:00,  1.00s/it]

Split 8  R² IN: 0.018892 | OOS: -0.022282 | SR IN: 0.546 | SR OOS: 0.512


PLS R² IN global : 0.01873
PLS R² OOS global : 0.02004


In [71]:
#PCR
r2_in_pcr, r2_oos_pcr = [], []
success_ratio_in_pcr, success_ratio_oos_pcr = [], []
feature_importance_pcr = []
best_components_pcr, mse_val_grids_pcr = [], []
df_in_pcr, df_oos_pcr = [], []

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):

    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])
    tickers_trainval = pd.concat([x_train["Ticker"], x_val["Ticker"]], ignore_index=True)
    dates_trainval = pd.concat([x_train["Date"], x_val["Date"]], ignore_index=True)

    assert len(x_trainval) == len(y_trainval) == len(tickers_trainval) == len(dates_trainval), f" Split {split_idx} — taille mismatch dans trainval"

    # Tuning
    candidate_ks = np.arange(1, 26)
    mse_val_grid = []
    best_mse = float('inf')
    best_k = None

    for k in candidate_ks:
        pcr_pipe = Pipeline([
            ('pca', PCA(n_components=k)),
            ('reg', LinearRegression())
        ])
        pcr_pipe.fit(x_train[covariates], y_train)
        y_val_pred = pcr_pipe.predict(x_val[covariates]).ravel()
        mse_val = mean_squared_error(y_val, y_val_pred)
        mse_val_grid.append(mse_val)
        if mse_val < best_mse:
            best_mse = mse_val
            best_k = k

    mse_val_grids_pcr.append(mse_val_grid)
    best_components_pcr.append(best_k)

    # Final fit
    pcr_final = Pipeline([
        ('pca', PCA(n_components=best_k)),
        ('reg', LinearRegression())
    ])
    pcr_final.fit(x_trainval, y_trainval)

    # Importance des variables
    reg_coef = pcr_final.named_steps['reg'].coef_  # (k,)
    pca_components = pcr_final.named_steps['pca'].components_  # (k, n_features)
    projected_coefs = np.abs(reg_coef @ pca_components)  # (n_features,)
    feature_importance_pcr.append(projected_coefs.flatten())

    # Prédictions
    y_trainval_pred = pcr_final.predict(x_trainval).ravel()
    y_test_pred = pcr_final.predict(x_test[covariates]).ravel()

    # Vérifications
    assert len(y_trainval_pred) == len(y_trainval), f" Split {split_idx} — pred trainval mismatch"
    assert len(y_test_pred) == len(y_test), f" Split {split_idx} — pred test mismatch"

    # Scores
    r2_in = r2(y_trainval, y_trainval_pred)
    r2_out = r2(y_test, y_test_pred)
    sr_in = success_ratio(y_trainval, y_trainval_pred)
    sr_out = success_ratio(y_test, y_test_pred)

    r2_in_pcr.append(r2_in)
    r2_oos_pcr.append(r2_out)
    success_ratio_in_pcr.append(sr_in)
    success_ratio_oos_pcr.append(sr_out)

    df_in_split = pd.DataFrame({
        "Date": dates_trainval,
        "Ticker": tickers_trainval,
        "y_true": y_trainval.values,
        "y_pred_pcr": y_trainval_pred,
        "Split": split_idx
    })
    df_in_pcr.append(df_in_split)

    df_test_split = pd.DataFrame({
        "Date": x_test["Date"].reset_index(drop=True),
        "Ticker": x_test["Ticker"].reset_index(drop=True),
        "y_true": y_test.reset_index(drop=True),
        "y_pred_pcr": y_test_pred
    })
    df_oos_pcr.append(df_test_split)

    print(f"Split {split_idx}  R² IN: {r2_in:.6f} | OOS: {r2_out:.6f} | SR IN: {sr_in:.3f} | SR OOS: {sr_out:.3f}")

# Concat final
df_in_pcr = pd.concat(df_in_pcr, ignore_index=True)
df_in_pcr.drop_duplicates(subset=["Date", "Ticker"], inplace=True)
df_oos_pcr = pd.concat(df_oos_pcr, ignore_index=True)

# Conversion
y_trainval_pcr = df_in_pcr["y_pred_pcr"].values
y_true_pcr = df_in_pcr["y_true"].values
y_pred_oos_pcr = df_oos_pcr["y_pred_pcr"].values
y_true_oos_pcr = df_oos_pcr["y_true"].values

assert len(y_trainval_pcr) == len(y_true_pcr), "Taille mismatch entre y_trainval_pcr et y_true_pcr"
assert len(y_pred_oos_pcr) == len(y_true_oos_pcr), "Taille mismatch entre y_pred_oos_pcr et y_true_oos_pcr"

print(f"PCR R² IN global : {r2(y_true_pcr, y_trainval_pcr):.5f}")
print(f"PCR R² OOS global : {r2(y_true_oos_pcr, y_pred_oos_pcr):.5f}")


 12%|█▎        | 1/8 [00:00<00:02,  3.23it/s]

Split 1  R² IN: 0.013605 | OOS: 0.150207 | SR IN: 0.540 | SR OOS: 0.706


 25%|██▌       | 2/8 [00:00<00:01,  3.37it/s]

Split 2  R² IN: 0.020176 | OOS: 0.015579 | SR IN: 0.548 | SR OOS: 0.520


 38%|███▊      | 3/8 [00:00<00:01,  3.20it/s]

...

 88%|████████▊ | 7/8 [00:02<00:00,  3.03it/s]

Split 7  R² IN: 0.020675 | OOS: -0.011664 | SR IN: 0.550 | SR OOS: 0.525


100%|██████████| 8/8 [00:02<00:00,  3.06it/s]

Split 8  R² IN: 0.017196 | OOS: -0.025900 | SR IN: 0.547 | SR OOS: 0.505


PCR R² IN global : 0.01873
PCR R² OOS global : 0.01949


In [72]:
#ENET
# ElasticNet : Elastic Net
# Hyperparamètres :
# - lambda (alpha) : coefficient de pénalisation choisi pour minimiser la MSE
# - l1_ratio fixé à 0.5

r2_in_en, r2_oos_en = [], []
success_ratio_in_en, success_ratio_oos_en = [], []
y_trainval_en = []

# Hyperparamètres spécifiques
best_lambdas = []
nonzero_counts_en = []
feature_importance_en = []

df_in_en, df_oos_en = [], []

enet_param_grid = {
    'alpha': np.logspace(-4, 0, num=10)
}

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):
    best_mse = float('inf')
    best_lambda = None

    # Recherche du meilleur alpha (lambda)
    for params in ParameterGrid(enet_param_grid):
        enet = ElasticNet(**params, l1_ratio=0.5, max_iter=10000)
        enet.fit(x_train[covariates], y_train)
        y_val_pred = enet.predict(x_val[covariates])
        mse = mean_squared_error(y_val, y_val_pred)
        if mse < best_mse:
            best_mse = mse
            best_lambda = params['alpha']

    best_lambdas.append(best_lambda)
    print(f"Split {split_idx} : meilleur lambda = {best_lambda}")

    # Entraînement final sur train+val
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])
    tickers_trainval = pd.concat([x_train["Ticker"], x_val["Ticker"]], ignore_index=True)
    dates_trainval = pd.concat([x_train["Date"], x_val["Date"]], ignore_index=True)

    assert len(x_trainval) == len(y_trainval) == len(tickers_trainval) == len(dates_trainval), f" Split {split_idx} — taille mismatch dans trainval"

    en_final = ElasticNet(alpha=best_lambda, l1_ratio=0.5, max_iter=10000)
    en_final.fit(x_trainval, y_trainval)

    y_trainval_pred = en_final.predict(x_trainval)
    y_test_pred = en_final.predict(x_test[covariates])

    assert len(y_trainval_pred) == len(y_trainval), f" Split {split_idx} — pred trainval mismatch"
    assert len(y_test_pred) == len(y_test), f" Split {split_idx} — pred test mismatch"

    # Importance et sparsité
    coefs = np.abs(en_final.coef_)
    feature_importance_en.append(coefs)
    nonzero_count = np.sum(en_final.coef_ != 0)
    nonzero_counts_en.append(nonzero_count)

    # Scores
    r2_in = r2(y_trainval, y_trainval_pred)
    r2_out = r2(y_test, y_test_pred)
    sr_in = success_ratio(y_trainval, y_trainval_pred)
    sr_out = success_ratio(y_test, y_test_pred)

    r2_in_en.append(r2_in)
    r2_oos_en.append(r2_out)
    success_ratio_in_en.append(sr_in)
    success_ratio_oos_en.append(sr_out)
    y_trainval_en.append(y_trainval_pred)

    # Dataframes
    df_in_split = pd.DataFrame({
        "Date": dates_trainval,
        "Ticker": tickers_trainval,
        "y_true": y_trainval.values,
        "y_pred_en": y_trainval_pred,
        "Split": split_idx
    })
    df_in_en.append(df_in_split)

    df_test_split = pd.DataFrame({
        "Date": x_test["Date"].reset_index(drop=True),
        "Ticker": x_test["Ticker"].reset_index(drop=True),
        "y_true": y_test.reset_index(drop=True),
        "y_pred_en": y_test_pred
    })
    df_oos_en.append(df_test_split)

    print(f"Split {split_idx}  R² IN: {r2_in:.6f} | OOS: {r2_out:.6f} | SR IN: {sr_in:.3f} | SR OOS: {sr_out:.3f}")

# Concat final
df_in_en = pd.concat(df_in_en, ignore_index=True)
df_in_en.drop_duplicates(subset=["Date", "Ticker"], inplace=True)
df_oos_en = pd.concat(df_oos_en, ignore_index=True)

# Conversion
y_trainval_en = df_in_en["y_pred_en"].values
y_true_en = df_in_en["y_true"].values
y_pred_oos_en = df_oos_en["y_pred_en"].values
y_true_oos_en = df_oos_en["y_true"].values

assert len(y_trainval_en) == len(y_true_en), "Taille mismatch entre y_trainval_en et y_true_en"
assert len(y_pred_oos_en) == len(y_true_oos_en), "Taille mismatch entre y_pred_oos_en et y_true_oos_en"

print(f"ENet R² IN global : {r2(y_true_en, y_trainval_en):.5f}")
print(f"ENet R² OOS global : {r2(y_true_oos_en, y_pred_oos_en):.5f}")


 12%|█▎        | 1/8 [00:00<00:00,  8.82it/s]

Split 1 : meilleur lambda = 0.000774263682681127
Split 1  R² IN: 0.015732 | OOS: 0.158783 | SR IN: 0.541 | SR OOS: 0.711


 25%|██▌       | 2/8 [00:00<00:00,  8.67it/s]

Split 2 : meilleur lambda = 0.000774263682681127
Split 2  R² IN: 0.019728 | OOS: 0.019641 | SR IN: 0.551 | SR OOS: 0.537


 38%|███▊      | 3/8 [00:00<00:00,  8.62it/s]

...

 88%|████████▊ | 7/8 [00:00<00:00,  7.79it/s]

Split 7 : meilleur lambda = 0.000774263682681127
Split 7  R² IN: 0.019654 | OOS: -0.007682 | SR IN: 0.553 | SR OOS: 0.522


100%|██████████| 8/8 [00:01<00:00,  7.06it/s]

Split 8 : meilleur lambda = 0.002154434690031882
Split 8  R² IN: 0.017206 | OOS: -0.024932 | SR IN: 0.553 | SR OOS: 0.498


ENet R² IN global : 0.01960
ENet R² OOS global : 0.02272


In [82]:
#RF

param_grid = {
    'n_estimators': [200, 400],       
    'max_depth': [3, 4, 5],
    'max_features': ['log2', 1, 2]
}



r2_in_rf, r2_oos_rf = [], []
success_ratio_in_rf, success_ratio_oos_rf = [], []
feature_importance_rf = []
df_in_rf, df_oos_rf = [], []
best_params_rf, mse_val_grids_rf = [], []
y_trainval_rf = []

# Grid Search + Entraînement
for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):
    best_mse = float('inf')
    best_params = None
    mse_grid = []

    for params in ParameterGrid(param_grid):
        rf = RandomForestRegressor(**params, n_jobs=-1, random_state=0)
        rf.fit(x_train[covariates], y_train)
        y_val_pred = rf.predict(x_val[covariates])
        mse = mean_squared_error(y_val, y_val_pred)
        mse_grid.append((params, mse))
        if mse < best_mse:
            best_mse = mse
            best_params = params

    mse_val_grids_rf.append(mse_grid)
    best_params_rf.append(best_params)
    print(f"\nSplit {split_idx} : meilleurs params = {best_params} (MSE val = {best_mse:.6f})")

    # Entraînement final
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])
    tickers_trainval = pd.concat([x_train["Ticker"], x_val["Ticker"]], ignore_index=True)
    dates_trainval = pd.concat([x_train["Date"], x_val["Date"]], ignore_index=True)

    rf_final = RandomForestRegressor(**best_params, n_jobs=-1, random_state=0)
    rf_final.fit(x_trainval, y_trainval)

    y_trainval_pred = rf_final.predict(x_trainval)
    y_test_pred = rf_final.predict(x_test[covariates])

    # Vérification
    assert len(x_trainval) == len(y_trainval) == len(tickers_trainval) == len(dates_trainval), f"❌ Split {split_idx} — taille mismatch dans trainval"
    assert len(y_trainval_pred) == len(y_trainval)
    assert len(y_test_pred) == len(y_test)

    # R² & Success Ratio
    r2_in = r2(y_trainval, y_trainval_pred)
    r2_out = r2(y_test, y_test_pred)
    sr_in = success_ratio(y_trainval, y_trainval_pred)
    sr_out = success_ratio(y_test, y_test_pred)

    r2_in_rf.append(r2_in)
    r2_oos_rf.append(r2_out)
    success_ratio_in_rf.append(sr_in)
    success_ratio_oos_rf.append(sr_out)
    feature_importance_rf.append(rf_final.feature_importances_)
    y_trainval_rf.append(y_trainval_pred)

    # DataFrames
    df_in_split = pd.DataFrame({
        "Date": dates_trainval,
        "Ticker": tickers_trainval,
        "y_true": y_trainval.values,
        "y_pred_rf": y_trainval_pred,
        "Split": split_idx
    })
    df_in_rf.append(df_in_split)

    df_test_split = pd.DataFrame({
        "Date": x_test["Date"].reset_index(drop=True),
        "Ticker": x_test["Ticker"].reset_index(drop=True),
        "y_true": y_test.reset_index(drop=True),
        "y_pred_rf": y_test_pred
    })
    df_oos_rf.append(df_test_split)

    print(f"Split {split_idx}  R² IN: {r2_in:.6f} | OOS: {r2_out:.6f} | SR IN: {sr_in:.3f} | SR OOS: {sr_out:.3f}")

# Concat final
df_in_rf = pd.concat(df_in_rf, ignore_index=True)
df_in_rf.drop_duplicates(subset=["Date", "Ticker"], inplace=True)
df_oos_rf = pd.concat(df_oos_rf, ignore_index=True)

# Conversion
y_trainval_rf = df_in_rf["y_pred_rf"].values
y_true_rf = df_in_rf["y_true"].values
y_pred_oos_rf = df_oos_rf["y_pred_rf"].values
y_true_oos_rf = df_oos_rf["y_true"].values

assert len(y_trainval_rf) == len(y_true_rf), "❌ Taille mismatch entre y_trainval_rf et y_true_rf"
assert len(y_pred_oos_rf) == len(y_true_oos_rf), "❌ Taille mismatch entre y_pred_oos_rf et y_true_oos_rf"

# Résumé global
print(f"\nRF — Shapes IN: {y_trainval_rf.shape} | OOS: {y_pred_oos_rf.shape}")
print(f"RF R² IN global : {r2(y_true_rf, y_trainval_rf):.5f}")
print(f"RF R² OOS global : {r2(y_true_oos_rf, y_pred_oos_rf):.5f}")
print(f"RF SR IN global : {success_ratio(y_true_rf, y_trainval_rf):.3f}")
print(f"RF SR OOS global : {success_ratio(y_true_oos_rf, y_pred_oos_rf):.3f}")


  0%|          | 0/8 [00:00<?, ?it/s]


Split 1 : meilleurs params = {'max_depth': 3, 'max_features': 'log2', 'n_estimators': 200} (MSE val = 0.005136)


 12%|█▎        | 1/8 [00:21<02:28, 21.28s/it]

Split 1  R² IN: 0.040524 | OOS: 0.147141 | SR IN: 0.548 | SR OOS: 0.730

Split 2 : meilleurs params = {'max_depth': 5, 'max_features': 'log2', 'n_estimators': 200} (MSE val = 0.003599)


 25%|██▌       | 2/8 [00:41<02:02, 20.50s/it]

...

 88%|████████▊ | 7/8 [02:21<00:19, 19.95s/it]

Split 7  R² IN: 0.050058 | OOS: -0.007082 | SR IN: 0.557 | SR OOS: 0.520

Split 8 : meilleurs params = {'max_depth': 5, 'max_features': 'log2', 'n_estimators': 200} (MSE val = 0.005116)


100%|██████████| 8/8 [02:41<00:00, 20.20s/it]

Split 8  R² IN: 0.092167 | OOS: -0.017133 | SR IN: 0.564 | SR OOS: 0.502



RF — Shapes IN: (9792,) | OOS: (3264,)
RF R² IN global : 0.04365
RF R² OOS global : 0.02294
RF SR IN global : 0.557
RF SR OOS global : 0.563


In [74]:
#GBRT
from sklearn.ensemble import GradientBoostingRegressor

r2_in_gbrt, r2_oos_gbrt = [], []
success_ratio_in_gbrt, success_ratio_oos_gbrt = [], []
feature_importance_gbrt, complexity_gbrt = [], []
df_in_gbrt, df_oos_gbrt = [], []
y_trainval_gbrt = []

best_params_gbrt = []
mse_val_grids_gbrt = []

param_grid_gbrt = {
    'n_estimators': [300],
    'learning_rate': [0.01],
    'max_depth': [2, 3, 4],
    'loss': ['huber'],
    'alpha': [0.9]
}

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):

    best_mse = float('inf')
    best_params = None
    mse_grid = []

    for params in ParameterGrid(param_grid_gbrt):
        model = GradientBoostingRegressor(**params, random_state=0)
        model.fit(x_train[covariates], y_train)
        y_val_pred = model.predict(x_val[covariates])
        mse = mean_squared_error(y_val, y_val_pred)
        mse_grid.append((params, mse))
        if mse < best_mse:
            best_mse = mse
            best_params = params

    mse_val_grids_gbrt.append(mse_grid)
    best_params_gbrt.append(best_params)
    print(f"Split {split_idx} — meilleurs params = {best_params} (MSE val = {best_mse:.6f})")

    # Train final
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])
    tickers_trainval = pd.concat([x_train["Ticker"], x_val["Ticker"]], ignore_index=True)
    dates_trainval = pd.concat([x_train["Date"], x_val["Date"]], ignore_index=True)

    assert len(x_trainval) == len(y_trainval) == len(tickers_trainval) == len(dates_trainval), f"Split {split_idx} — mismatch tailles trainval"

    gbrt_final = GradientBoostingRegressor(**best_params, random_state=0)
    gbrt_final.fit(x_trainval, y_trainval)

    y_trainval_pred = gbrt_final.predict(x_trainval)
    y_test_pred = gbrt_final.predict(x_test[covariates])

    assert len(y_trainval_pred) == len(y_trainval), f"Split {split_idx} — mismatch pred trainval"
    assert len(y_test_pred) == len(y_test), f"Split {split_idx} — mismatch pred test"

    # Scores
    r2_in = r2(y_trainval, y_trainval_pred)
    r2_out = r2(y_test, y_test_pred)
    sr_in = success_ratio(y_trainval, y_trainval_pred)
    sr_out = success_ratio(y_test, y_test_pred)

    r2_in_gbrt.append(r2_in)
    r2_oos_gbrt.append(r2_out)
    success_ratio_in_gbrt.append(sr_in)
    success_ratio_oos_gbrt.append(sr_out)

    # Feature importance
    importances = gbrt_final.feature_importances_
    feature_importance_gbrt.append(importances)
    complexity_gbrt.append(np.sum(importances > 0))

    # DataFrames
    df_in_split = pd.DataFrame({
        "Date": dates_trainval,
        "Ticker": tickers_trainval,
        "y_true": y_trainval.values,
        "y_pred_gbrt": y_trainval_pred,
        "Split": split_idx
    })
    df_in_gbrt.append(df_in_split)

    df_test_split = pd.DataFrame({
        "Date": x_test["Date"].reset_index(drop=True),
        "Ticker": x_test["Ticker"].reset_index(drop=True),
        "y_true": y_test.reset_index(drop=True),
        "y_pred_gbrt": y_test_pred
    })
    df_oos_gbrt.append(df_test_split)

    print(f"Split {split_idx}  R² IN: {r2_in:.6f} | OOS: {r2_out:.6f} | SR IN: {sr_in:.3f} | SR OOS: {sr_out:.3f}")

# Concat
df_in_gbrt = pd.concat(df_in_gbrt, ignore_index=True)
df_in_gbrt.drop_duplicates(subset=["Date", "Ticker"], inplace=True)
df_oos_gbrt = pd.concat(df_oos_gbrt, ignore_index=True)

# Conversion
y_trainval_gbrt = df_in_gbrt["y_pred_gbrt"].values
y_true_gbrt = df_in_gbrt["y_true"].values
y_pred_oos_gbrt = df_oos_gbrt["y_pred_gbrt"].values
y_true_oos_gbrt = df_oos_gbrt["y_true"].values

assert len(y_trainval_gbrt) == len(y_true_gbrt), "Taille mismatch entre y_trainval_gbrt et y_true_gbrt"
assert len(y_pred_oos_gbrt) == len(y_true_oos_gbrt), "Taille mismatch entre y_pred_oos_gbrt et y_true_oos_gbrt"

print(f"GBRT R² IN global : {r2(y_true_gbrt, y_trainval_gbrt):.5f}")
print(f"GBRT R² OOS global : {r2(y_true_oos_gbrt, y_pred_oos_gbrt):.5f}")


  0%|          | 0/8 [00:00<?, ?it/s]

Split 1 — meilleurs params = {'alpha': 0.9, 'learning_rate': 0.01, 'loss': 'huber', 'max_depth': 2, 'n_estimators': 300} (MSE val = 0.005204)


 12%|█▎        | 1/8 [00:32<03:46, 32.42s/it]

Split 1  R² IN: 0.028076 | OOS: 0.135523 | SR IN: 0.557 | SR OOS: 0.718
Split 2 — meilleurs params = {'alpha': 0.9, 'learning_rate': 0.01, 'loss': 'huber', 'max_depth': 2, 'n_estimators': 300} (MSE val = 0.003678)


 25%|██▌       | 2/8 [01:05<03:16, 32.83s/it]

...

 88%|████████▊ | 7/8 [04:50<00:43, 43.52s/it]

Split 7  R² IN: 0.030691 | OOS: -0.003314 | SR IN: 0.559 | SR OOS: 0.525
Split 8 — meilleurs params = {'alpha': 0.9, 'learning_rate': 0.01, 'loss': 'huber', 'max_depth': 2, 'n_estimators': 300} (MSE val = 0.005128)


100%|██████████| 8/8 [05:37<00:00, 42.15s/it]

Split 8  R² IN: 0.028880 | OOS: -0.018797 | SR IN: 0.558 | SR OOS: 0.505


GBRT R² IN global : 0.03323
GBRT R² OOS global : 0.01804


In [83]:
#XGBOOST
# Initialisations
r2_in_xgb, r2_oos_xgb = [], []
success_ratio_in_xgb, success_ratio_oos_xgb = [], []
feature_importance_xgb = []
df_in_xgb, df_oos_xgb = [], []
y_trainval_xgb = []

# Grille de recherche
param_grid = {
    'max_depth': [2, 3],
    'learning_rate': [0.01],
    'n_estimators': [200, 400, 800],
}

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):

    # Recherche du meilleur modèle (validation)
    best_mse = float('inf')
    best_params = None

    for params in ParameterGrid(param_grid):
        model = XGBRegressor(**params, random_state=0, n_jobs=-1)
        model.fit(x_train[covariates], y_train)
        preds = model.predict(x_val[covariates])
        mse = mean_squared_error(y_val, preds)

        if mse < best_mse:
            best_mse = mse
            best_params = params

    print(f"Split {split_idx} : meilleurs params = {best_params} (MSE val = {best_mse:.6f})")

    # Réentraînement sur train + val
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])
    tickers_trainval = pd.concat([x_train["Ticker"], x_val["Ticker"]], ignore_index=True)
    dates_trainval = pd.concat([x_train["Date"], x_val["Date"]], ignore_index=True)

    model = XGBRegressor(**best_params, random_state=0, n_jobs=-1)
    model.fit(x_trainval, y_trainval)

    y_trainval_pred = model.predict(x_trainval)
    y_test_pred = model.predict(x_test[covariates])

    # Vérifications
    assert len(y_trainval_pred) == len(y_trainval)
    assert len(y_test_pred) == len(y_test)

    # Scores
    r2_in = r2(y_trainval, y_trainval_pred)
    r2_out = r2(y_test, y_test_pred)
    sr_in = success_ratio(y_trainval, y_trainval_pred)
    sr_out = success_ratio(y_test, y_test_pred)

    r2_in_xgb.append(r2_in)
    r2_oos_xgb.append(r2_out)
    success_ratio_in_xgb.append(sr_in)
    success_ratio_oos_xgb.append(sr_out)

    # Importance des variables
    importances = model.feature_importances_
    feature_importance_xgb.append(importances)

    # Prédictions in-sample
    df_in_split = pd.DataFrame({
        "Date": dates_trainval,
        "Ticker": tickers_trainval,
        "y_true": y_trainval.values,
        "y_pred_xgb": y_trainval_pred,
        "Split": split_idx
    })
    df_in_xgb.append(df_in_split)

    # Prédictions OOS
    df_test_split = pd.DataFrame({
        "Date": x_test["Date"].reset_index(drop=True),
        "Ticker": x_test["Ticker"].reset_index(drop=True),
        "y_true": y_test.reset_index(drop=True),
        "y_pred_xgb": y_test_pred
    })
    df_oos_xgb.append(df_test_split)

    print(f"Split {split_idx}  R² IN: {r2_in:.6f} | OOS: {r2_out:.6f} | SR IN: {sr_in:.3f} | SR OOS: {sr_out:.3f}")

# Concat final
df_in_xgb = pd.concat(df_in_xgb, ignore_index=True)
df_in_xgb.drop_duplicates(subset=["Date", "Ticker"], inplace=True)
df_oos_xgb = pd.concat(df_oos_xgb, ignore_index=True)

# Conversion
y_trainval_xgb = df_in_xgb["y_pred_xgb"].values
y_true_xgb = df_in_xgb["y_true"].values
y_pred_oos_xgb = df_oos_xgb["y_pred_xgb"].values
y_true_oos_xgb = df_oos_xgb["y_true"].values

assert len(y_trainval_xgb) == len(y_true_xgb)
assert len(y_pred_oos_xgb) == len(y_true_oos_xgb)

# Résumés globaux
print(f"XGB R² IN global : {r2(y_true_xgb, y_trainval_xgb):.5f}")
print(f"XGB R² OOS global : {r2(y_true_oos_xgb, y_pred_oos_xgb):.5f}")


  0%|          | 0/8 [00:00<?, ?it/s]

Split 1 : meilleurs params = {'learning_rate': 0.01, 'max_depth': 2, 'n_estimators': 200} (MSE val = 0.005186)


 12%|█▎        | 1/8 [00:08<01:01,  8.80s/it]

Split 1  R² IN: 0.034398 | OOS: 0.145457 | SR IN: 0.548 | SR OOS: 0.728
Split 2 : meilleurs params = {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 200} (MSE val = 0.003637)


 25%|██▌       | 2/8 [00:19<00:58,  9.68s/it]

...

 88%|████████▊ | 7/8 [01:43<00:16, 16.43s/it]

Split 7  R² IN: 0.032153 | OOS: -0.010058 | SR IN: 0.556 | SR OOS: 0.517
Split 8 : meilleurs params = {'learning_rate': 0.01, 'max_depth': 2, 'n_estimators': 200} (MSE val = 0.005164)


100%|██████████| 8/8 [01:55<00:00, 14.41s/it]

Split 8  R² IN: 0.030204 | OOS: -0.018478 | SR IN: 0.554 | SR OOS: 0.502


XGB R² IN global : 0.03729
XGB R² OOS global : 0.01774


III RESULTS

In [76]:
#vérif y true / ytraintrue
y_true_list = [
    y_true_oos_ols,
    y_true_oos_pls,
    y_true_oos_pcr,
    y_true_oos_en,
    y_true_oos_rf,
    y_true_oos_gbrt,
    y_true_oos_xgb
]

y_true_trainval_list = [
    y_true_ols,
    y_true_pls,
    y_true_pcr,
    y_true_en,
    y_true_rf,
    y_true_gbrt,
    y_true_xgb
]

# Comparaison entre le premier et tous les autres
all_equal = all(np.array_equal(y_true_list[0], y) for y in y_true_list[1:])

if all_equal:
    print("identique")
else:
    print("erreur → différence entre les listes")

all_equal_y_true_trainval = all(np.array_equal(y_true_trainval_list[0], y) for y in y_true_trainval_list[1:])

if all_equal_y_true_trainval:
    print("identique")
else:
    print("erreur → différence entre les listes")

identique
identique


In [77]:
#ils sont identiques → on en garde qu'un
y_true_trainval = y_true_ols
y_true_oos = y_true_oos_ols

In [78]:
#R2 calcul

models = ["HA", "OLS", "PLS", "PCR", "Enet", "RF", "GBRT", "XGB"]

y_pred_trainval = {
    "OLS": y_trainval_ols,
    "PLS": y_trainval_pls,
    "PCR": y_trainval_pcr,
    "Enet": y_trainval_en,
    "RF": y_trainval_rf,
    "GBRT": y_trainval_gbrt,
    "XGB": y_trainval_xgb,
    "HA": y_trainval_ha
}

y_pred_oos = {
    "OLS": y_pred_oos_ols,
    "PLS": y_pred_oos_pls,
    "PCR": y_pred_oos_pcr,
    "Enet": y_pred_oos_en,
    "RF": y_pred_oos_rf,
    "GBRT": y_pred_oos_gbrt,
    "XGB": y_pred_oos_xgb,
    "HA": y_true_ha  
}


#calculs des métriques
rows_in, rows_oos = [], []
#in-sample 
for model_name, y_pred in y_pred_trainval.items():
    r2_in_val = r2(y_true_trainval, y_pred)
    r2_in_vs_ha_val = r2_vs_benchmark(y_true_trainval, y_pred, y_pred_trainval["HA"])
    sr_in_val = success_ratio(y_true_trainval, y_pred)

    rows_in.append({
        "Model": model_name,
        "In-sample $R^2$": r2_in_val,
        "In-sample $R^2$ vs HA": r2_in_vs_ha_val,
        "Success Ratio (In-sample)": sr_in_val
    })

df_results_in = pd.DataFrame(rows_in)

#OOS

for model_name in y_pred_oos.keys():
    r2_oos_val = r2(y_true_oos, y_pred_oos[model_name])
    r2_oos_vs_ha_val = r2_vs_benchmark(y_true_oos, y_pred_oos[model_name], y_pred_oos["HA"])
    sr_oos_val = success_ratio(y_true_oos, y_pred_oos[model_name])

    rows_oos.append({
        "Model": model_name,
        "Out-of-sample $R^2$": r2_oos_val,
        "Out-of-sample $R^2$ vs HA": r2_oos_vs_ha_val,
        "Success Ratio (OOS)": sr_oos_val
    })

df_results_oos = pd.DataFrame(rows_oos)


In [84]:
#export latex 

#df in sample
df_r2_in_formatted = df_results_in.copy()

# Formatage en pourcentage avec 2 décimales
for col in df_r2_in_formatted.columns:
    if col != "Model":
        df_r2_in_formatted[col] = (
            df_r2_in_formatted[col].astype(float) * 100
        ).apply(lambda x: f"{x:.2f}")

# Export LaTeX
latex_r2_in = df_r2_in_formatted.to_latex(index=False, escape=False, column_format="lccc")
print(latex_r2_in)

#R2 oos 
df_r2_oos = df_results_oos[["Model", "Out-of-sample $R^2$"]].copy()
df_r2_oos["Out-of-sample $R^2$"] = (df_r2_oos["Out-of-sample $R^2$"].astype(float) * 100).apply(lambda x: f"{x:.2f}")
latex_r2_oos = df_r2_oos.to_latex(index=False, escape=False, column_format="lcc")
print(latex_r2_oos)

#R2 oos benchmark
df_r2_oos_benchmark = df_results_oos[["Model", "Out-of-sample $R^2$ vs HA", "Success Ratio (OOS)"]].copy()
df_r2_oos_benchmark["Out-of-sample $R^2$ vs HA"] = (df_r2_oos_benchmark["Out-of-sample $R^2$ vs HA"].astype(float) * 100).apply(lambda x: f"{x:.3f}")
df_r2_oos_benchmark["Success Ratio (OOS)"] = (df_r2_oos_benchmark["Success Ratio (OOS)"].astype(float) * 100).apply(lambda x: f"{x:.3f}")
latex_r2_benchmark = df_r2_oos_benchmark.to_latex(index=False, escape=False, column_format="lcc")
print(latex_r2_benchmark)


\begin{tabular}{lccc}
\toprule
Model & In-sample $R^2$ & In-sample $R^2$ vs HA & Success Ratio (In-sample) \\
\midrule
OLS & 2.09 & nan & 54.45 \\
PLS & 1.87 & nan & 54.60 \\
PCR & 1.87 & nan & 54.73 \\
Enet & 1.96 & nan & 54.89 \\
RF & 4.78 & nan & 55.98 \\
GBRT & 3.32 & nan & 56.50 \\
XGB & 3.73 & nan & 55.70 \\
HA & nan & nan & 55.12 \\
\bottomrule
\end{tabular}

\begin{tabular}{lcc}
\toprule
Model & Out-of-sample $R^2$ \\
\midrule
OLS & 1.81 \\
PLS & 2.00 \\
PCR & 1.95 \\
Enet & 2.27 \\
RF & 2.12 \\
GBRT & 1.80 \\
XGB & 1.77 \\
HA & 1.56 \\
\bottomrule
\end{tabular}

\begin{tabular}{lcc}
\toprule
Model & Out-of-sample $R^2$ vs HA & Success Ratio (OOS) \\
\midrule
OLS & 0.249 & 55.287 \\
PLS & 0.448 & 55.440 \\
PCR & 0.392 & 55.470 \\
Enet & 0.721 & 55.869 \\
RF & 0.562 & 56.206 \\
GBRT & 0.245 & 56.237 \\
XGB & 0.214 & 56.022 \\
HA & 0.000 & 56.298 \\
\bottomrule
\end{tabular}



In [80]:
df_oos = pd.DataFrame(y_pred_oos)
df_oos["y_true"] = y_true_oos  # ajoute la colonne avec les vraies valeurs


#DIABOLD TEST 
models = [col for col in df_oos.columns if col != "y_true"]


results = []

for i in range(len(models)):
    for j in range(i+1, len(models)):
        model1 = models[i]
        model2 = models[j]

        #calcule mse du modèle 1 et 2 
        e1 = (df_oos['y_true'] - df_oos[model1])**2
        e2 = (df_oos['y_true'] - df_oos[model2])**2
        d = e1 - e2
        d = d.dropna()

        X = np.ones(len(d))  # régression constante
        model = sm.OLS(d, X).fit()
        cov = cov_hac(model, nlags=1)  
        se = np.sqrt(cov[0][0])
        dm_stat = d.mean() / se
        p_value = 2 * (1 - t.cdf(abs(dm_stat), df=len(d) - 1))

        results.append({
            'Model 1': model1,
            'Model 2': model2,
            'DM Stat': dm_stat,
            'P-Value': p_value,
            'Best Model': model2 if dm_stat > 0 else model1
        })


dm_df = pd.DataFrame(results)
dm_df_sorted = dm_df.sort_values(by="P-Value", ascending=True)
#print(dm_df_sorted)
print(dm_df)

   Model 1 Model 2   DM Stat   P-Value Best Model
0      OLS     PLS  0.818968  0.412865        PLS
1      OLS     PCR  0.504908  0.613658        PCR
2      OLS    Enet  1.765746  0.077532       Enet
3      OLS      RF  0.826114  0.408800         RF
4      OLS    GBRT -0.007917  0.993684        OLS
5      OLS     XGB -0.087748  0.930082        OLS
6      OLS      HA -0.558833  0.576314        OLS
7      PLS     PCR -0.413651  0.679157        PLS
8      PLS    Enet  1.463922  0.143311       Enet
9      PLS      RF  0.389006  0.697297         RF
10     PLS    GBRT -0.563203  0.573335        PLS
11     PLS     XGB -0.731133  0.464751        PLS
12     PLS      HA -1.256169  0.209145        PLS
13     PCR    Enet  1.788801  0.073740       Enet
14     PCR      RF  0.577859  0.563399         RF
15     PCR    GBRT -0.416561  0.677027        PCR
16     PCR     XGB -0.561270  0.574652        PCR
17     PCR      HA -1.193770  0.232655        PCR
18    Enet      RF -0.661985  0.508028       Enet
